In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_8.py ──
"""
Shared infrastructure for MLFP04 Exercise 8 — Deep Learning Foundations.

Contains: synthetic XOR data, synthetic Singapore-medical image data,
reusable training loops, gradient monitoring helpers, ModelVisualizer
output paths. Technique-specific code (model classes, per-file training
loops, scenario narratives) does NOT belong here — it lives per file.
"""

from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from kailash_ml import ModelVisualizer


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT — seeds, device, output dir
# ════════════════════════════════════════════════════════════════════════
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex8_deep_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared hyperparameters
N_FEATS_XOR = 4
N_XOR_SAMPLES = 200
N_IMG_SAMPLES = 5000
IMG_SIZE = 64
N_CHANNELS = 1
N_CLASSES = 5
BATCH_SIZE = 64

# Kailash visualiser (used by every phase 4 block)
viz = ModelVisualizer()

# ════════════════════════════════════════════════════════════════════════
# DATA — XOR toy problem (Tasks 1-3)
# ════════════════════════════════════════════════════════════════════════


def make_xor_data(
    n_samples: int = N_XOR_SAMPLES, n_features: int = N_FEATS_XOR, seed: int = 42
) -> tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Generate a synthetic XOR classification task.

    Label is XOR of the sign of features 0 and 1. Features 2..n-1 are noise.
    Returns (X_tensor, y_tensor, y_numpy) on CPU.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, n_features)).astype(np.float32)
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(np.float32)
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y).unsqueeze(1)
    return X_t, y_t, y


# ════════════════════════════════════════════════════════════════════════
# DATA — Synthetic Singapore Hospital imaging tensors (Tasks 4-10)
# ════════════════════════════════════════════════════════════════════════
# Scenario: NUH (National University Hospital) chest-film triage. The real
# pipeline uses anonymised 512x512 DICOMs; this exercise uses 64x64 random
# tensors with the same multi-label structure so training completes in
# minutes on a laptop CPU / Colab T4.

SG_HOSPITAL_CLASSES = [
    "pneumonia",
    "effusion",
    "atelectasis",
    "nodule",
    "normal",
]


def make_sg_imaging_data(
    n_samples: int = N_IMG_SAMPLES, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_images, y_labels) as float32 numpy arrays.

    X: (N, 1, 64, 64) — simulated single-channel chest film tensors.
    y: (N, 5) — multi-label (~15% positive per class).
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, N_CHANNELS, IMG_SIZE, IMG_SIZE)).astype(
        np.float32
    )
    y = (rng.random((n_samples, N_CLASSES)) > 0.85).astype(np.float32)
    return X, y


def build_sg_loaders(
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, np.ndarray, np.ndarray]:
    """Produce (train_loader, test_loader, X_test_np, y_test_np) for the CNN tasks."""
    X, y = make_sg_imaging_data()
    split = int(0.8 * len(X))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader, X_te, y_te


# ════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════


def train_xor_net(
    net: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    optimiser: torch.optim.Optimizer,
    n_epochs: int = 100,
    criterion: nn.Module | None = None,
) -> list[float]:
    """Fit a small binary classifier to XOR data. Returns per-epoch loss."""
    crit = criterion or nn.BCEWithLogitsLoss()
    losses: list[float] = []
    for _ in range(n_epochs):
        optimiser.zero_grad()
        loss = crit(net(X), y)
        loss.backward()
        optimiser.step()
        losses.append(loss.item())
    return losses


def xor_accuracy(net: nn.Module, X: torch.Tensor, y_np: np.ndarray) -> float:
    """Binary accuracy on XOR data (threshold at 0.5)."""
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(X)).numpy().flatten()
    return float(((probs > 0.5) == y_np).mean())


def grad_norm(model: nn.Module) -> float:
    """L2 norm of the concatenated gradient vector."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return float(total**0.5)


def train_cnn_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    criterion: nn.Module,
    clip_value: float | None = None,
) -> tuple[float, float]:
    """Train for one epoch on the Singapore imaging loader.

    Returns (mean_loss, mean_grad_norm). If ``clip_value`` is set, the grad
    norm is measured pre-clipping and ``clip_grad_norm_`` is applied.
    """
    model.train()
    losses: list[float] = []
    grads: list[float] = []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        grads.append(grad_norm(model))
        if clip_value is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimiser.step()
        losses.append(loss.item())
    return float(np.mean(losses)), float(np.mean(grads))


def eval_cnn(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> float:
    """Return mean validation loss across the loader."""
    model.eval()
    losses: list[float] = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            losses.append(criterion(model(X_b), y_b).item())
    return float(np.mean(losses))


# ════════════════════════════════════════════════════════════════════════
# CNN BUILDING BLOCKS (reused across files 03, 04, 05)
# ════════════════════════════════════════════════════════════════════════


class ResBlock(nn.Module):
    """Residual block: skip connection preserves gradient flow."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + residual)


class TriageCNN(nn.Module):
    """CNN for multi-label Singapore hospital triage.

    Architecture: Conv32 -> ResBlock -> Conv64 -> ResBlock -> AdaptiveAvgPool
    -> Dropout -> Linear. Designed for the multi-label BCEWithLogitsLoss
    setup used throughout Exercise 8.
    """

    def __init__(self, n_classes: int = N_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(64),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        return self.classifier(self.features(x))


def count_params(model: nn.Module) -> tuple[int, int]:
    """Return (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ════════════════════════════════════════════════════════════════════════
# DATA LOADER ENTRY POINT
# ════════════════════════════════════════════════════════════════════════
# We expose an MLFPDataLoader handle so student files have a single import
# path even though the tensors are generated on the fly. Real datasets for
# CNN fine-tuning live in Module 5.
loader = MLFPDataLoader()




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 8.4: Optimisers and Learning-Rate Schedulers
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compare SGD, SGD+momentum, Adam, and AdamW on the same task
#   - Apply a cosine-annealing LR schedule with a linear warmup
#   - Track LR and loss together during training
#
# PREREQUISITES: 03_cnn_residual.py
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — optimisers as adaptive gradient rescaling
#   2. Build — optimiser grid and a warmup+cosine scheduler
#   3. Train — short runs + one full run with the scheduler
#   4. Visualise — optimiser curves + schedule trajectory
#   5. Apply — Sea Group fraud model: warmup saved a production rollout
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.optim as optim


print("\n" + "=" * 70)
print("  Optimisers and Schedulers — How Fast Can You Converge?")
print("=" * 70)



## THEORY — Optimisers as per-parameter learning rates


In [ ]:
# Adam normalises each parameter by its own gradient variance. AdamW
# fixes weight decay to behave as a true regulariser. Cosine annealing
# reduces LR smoothly; linear warmup prevents Adam from blowing up
# before its variance estimates have stabilised.

train_loader, test_loader, _, _ = build_sg_loaders()
criterion = nn.BCEWithLogitsLoss()
n_classes = len(SG_HOSPITAL_CLASSES)


def fresh_model() -> TriageCNN:
    return TriageCNN(n_classes=n_classes, dropout_rate=0.3).to(device)



## TASK 2 — BUILD optimiser grid + scheduler factory


Linear warmup then cosine decay to zero.


In [ ]:
# TODO: Fill in the missing lambdas so each value is a callable taking
# `params` and returning the configured optimiser.
optimiser_builders: dict[str, callable] = {
    "SGD lr=0.01": lambda p: optim.SGD(p, lr=0.01),
    "SGD+momentum": lambda p: ____,  # SGD lr=0.01, momentum=0.9
    "Adam lr=1e-3": lambda p: ____,  # Adam lr=1e-3
    "AdamW lr=1e-3": lambda p: ____,  # AdamW lr=1e-3, weight_decay=1e-4
}


def make_warmup_cosine(
    optimiser: optim.Optimizer, warmup_epochs: int, total_epochs: int
) -> optim.lr_scheduler.LambdaLR:

    def lr_lambda(epoch: int) -> float:
        # TODO: For the first `warmup_epochs` epochs return a linearly
        # increasing multiplier (epoch+1)/warmup_epochs. After that,
        # return 0.5 * (1 + cos(pi * progress)) where progress is
        # (epoch - warmup_epochs) / (total_epochs - warmup_epochs).
        if epoch < warmup_epochs:
            return ____
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return ____

    return optim.lr_scheduler.LambdaLR(optimiser, lr_lambda=lr_lambda)



## TASK 3 — TRAIN the optimiser grid (5 epochs each)


In [ ]:
print("\n--- Optimiser comparison (5 epochs each) ---")
optimiser_histories: dict[str, list[float]] = {}
for name, builder in optimiser_builders.items():
    model = fresh_model()
    optimiser = builder(model.parameters())
    losses: list[float] = []
    for _ in range(5):
        loss, _ = train_cnn_one_epoch(model, train_loader, optimiser, criterion)
        losses.append(loss)
    optimiser_histories[name] = losses
    print(f"  {name:<18}: {losses[0]:.4f} -> {losses[-1]:.4f}")

assert all(
    h[-1] <= h[0] + 1e-3 for h in optimiser_histories.values()
), "Task 3: every optimiser should hold the line on train loss"

# Full run with scheduler
print("\n--- Full run: AdamW + warmup(2) + cosine(10) ---")
total_epochs = 10
warmup_epochs = 2

model = fresh_model()
# TODO: Create an AdamW optimiser with lr=1e-3, weight_decay=1e-4.
optimiser = ____
# TODO: Create the warmup+cosine scheduler via make_warmup_cosine.
scheduler = ____

schedule_history = {"train_loss": [], "val_loss": [], "lr": []}
for epoch in range(total_epochs):
    train_loss, _ = train_cnn_one_epoch(model, train_loader, optimiser, criterion)
    val_loss = eval_cnn(model, test_loader, criterion)
    schedule_history["train_loss"].append(train_loss)
    schedule_history["val_loss"].append(val_loss)
    schedule_history["lr"].append(scheduler.get_last_lr()[0])
    scheduler.step()
    print(
        f"  Epoch {epoch + 1:>2}/{total_epochs}: "
        f"train={train_loss:.4f} val={val_loss:.4f} "
        f"lr={schedule_history['lr'][-1]:.6f}"
    )

assert (
    schedule_history["lr"][-1] < schedule_history["lr"][warmup_epochs]
), "Task 3: cosine tail should be lower than warmup peak"
assert (
    schedule_history["lr"][0] < schedule_history["lr"][warmup_epochs]
), "Task 3: warmup should start below the peak LR"
print("\n[ok] Checkpoint passed — warmup + cosine behave as specified\n")



## TASK 4 — VISUALISE


In [ ]:
fig_opt = viz.training_history(optimiser_histories, x_label="Epoch")
fig_opt.update_layout(title="Optimiser Comparison on TriageCNN")
fig_opt.write_html(OUTPUT_DIR / "04_optimiser_curves.html")

fig_sched = viz.training_history(
    {
        "Train BCE": schedule_history["train_loss"],
        "Val BCE": schedule_history["val_loss"],
        "Learning Rate (x1e3)": [lr * 1000 for lr in schedule_history["lr"]],
    },
    x_label="Epoch",
)
fig_sched.update_layout(title="Warmup(2) + Cosine(10) — Loss and LR Together")
fig_sched.write_html(OUTPUT_DIR / "04_schedule_curves.html")
print("[viz] 04_optimiser_curves.html + 04_schedule_curves.html saved")



## TASK 5 — APPLY: Sea Group (Shopee) Fraud Scoring


In [ ]:
# SCENARIO: Shopee Pay's v1 fraud scorer (SGD lr=0.1) diverged ~1 night
# in 10 during the hour-3 loss phase, triggering aborted rollouts worth
# ~S$1,200 each plus 4am pages. Switching to AdamW + 500-step warmup +
# cosine decay produced 0 aborted runs across 180 subsequent training
# nights.
#
# BUSINESS IMPACT: ~S$18K/month recovered aborted-run cost, ~60 eng-hours
# of on-call saved, enabling safe deployment of a S$140M/year fraud model.



## REFLECTION


[x] Compared SGD, SGD+momentum, Adam, AdamW on identical budgets
  [x] Built a linear-warmup + cosine-annealing scheduler
  [x] Trained a CNN for 10 epochs tracking train/val/LR together
  [x] Connected the pattern to Sea Group's production rollout

  Next: 05_regularisation_training.py — dropout, BN, clip, early stop, ONNX.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

